# 3.0 - HOG + SVM

Extração de características HOG a partir das imagens já processadas pelo
MTCNN, usando os parâmetros padrão do descritor propostos por Dalal e
Triggs (2005). As features são usadas para treinar um classificador SVM.

O modelo é avaliado primeiro no conjunto de validação, e o resultado
final reportado  é obtido avaliando o modelo uma única vez no
conjunto de teste.

In [ ]:
# Clona o repositorio e instala as dependencias.
# facenet-pytorch precisa de --no-deps porque o Colab nao tem wheels
# pre-compiladas pras versoes antigas de numpy/Pillow que ele pede.
# (Esse notebook nao roda MTCNN, mas mantemos o mesmo setup dos outros
# notebooks pra evitar erro de dependencia faltando.)

!git clone https://github.com/laianemuckler/liveness-detection.git
%cd liveness-detection

!pip install -r requirements.txt --quiet
!pip install facenet-pytorch==2.6.0 --no-deps --quiet


In [ ]:
# Monta o Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Importa as funções do projeto

import os
import numpy as np
import pandas as pd

from src.config import TRAIN_DIR, VAL_DIR, TEST_DIR, DRIVE_ROOT
from src.dataset import load_processed_split
from src.features import extract_features
from src.modeling.train import train_svm
from src.modeling.predict import evaluate, log_experiment

In [ ]:
# Carrega as imagens e labels pré-processadas com MTCNN

train_paths, y_train = load_processed_split(TRAIN_DIR)
val_paths, y_val = load_processed_split(VAL_DIR)
test_paths, y_test = load_processed_split(TEST_DIR)

print(f"Train: {len(train_paths)} | Validation: {len(val_paths)} | Test: {len(test_paths)}")

In [ ]:
# Cria os diretórios necessários

RUNS_DIR = os.path.join(DRIVE_ROOT, 'experiments', 'hog_svm', 'runs')
os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
# Extrai as features HOG padrão (Dalal & Triggs, 2005)
# ou carrega do disco, se ja foram extraidas numa execucao anterior

EXP_ID = 'hog_svm_01_default'
exp_dir = os.path.join(RUNS_DIR, EXP_ID)
os.makedirs(exp_dir, exist_ok=True)

train_feat_path = os.path.join(exp_dir, 'X_train.npy')
val_feat_path = os.path.join(exp_dir, 'X_val.npy')
test_feat_path = os.path.join(exp_dir, 'X_test.npy')

if os.path.exists(train_feat_path):
    print("Features ja extraidas, carregando do disco...")
    X_train = np.load(train_feat_path)
    X_val = np.load(val_feat_path)
    X_test = np.load(test_feat_path)
else:
    print("Extraindo features HOG...")
    X_train = extract_features(train_paths, 'hog_default')
    X_val = extract_features(val_paths, 'hog_default')
    X_test = extract_features(test_paths, 'hog_default')

    np.save(train_feat_path, X_train)
    np.save(val_feat_path, X_val)
    np.save(test_feat_path, X_test)

print(f"Formato das features: {X_train.shape}")

In [ ]:
# Treina o SVM usando as features extraidas do conjunto de treino.

model, scaler = train_svm(X_train, y_train, kernel='rbf', C=1.0)

In [ ]:
# Avalia o modelo no conjunto de validação e loga os resultados

val_results = evaluate(model, scaler, X_val, y_val)
print(f"Validation -> HTER: {val_results['HTER']*100:.2f}% | AUC: {val_results['AUC']:.4f}")

log_experiment(
    exp_id=EXP_ID,
    metodo='HOG+SVM',
    feature_config='hog_default',
    modelo_config='kernel=rbf, C=1.0',
    hter_val=val_results['HTER'],
    auc_val=val_results['AUC'],
)

In [ ]:
# avaliação no conjunto de teste

test_results = evaluate(model, scaler, X_test, y_test)

print(f"FINAL TEST RESULT ({EXP_ID})")
print(f"HTER: {test_results['HTER']*100:.2f}%")
print(f"AUC:  {test_results['AUC']:.4f}")

log_experiment(
    exp_id=EXP_ID + '_FINAL_TEST',
    metodo='HOG+SVM',
    feature_config='hog_default',
    modelo_config='kernel=rbf, C=1.0',
    hter_test=test_results['HTER'],
    auc_test=test_results['AUC'],
    obs='Final reported test metric for HOG+SVM'
)

In [ ]:
# Salva as predições para posterir analise

df_predictions = pd.DataFrame({
    'image_path': test_paths,
    'label_true': y_test,
    'label_pred': test_results['y_pred'],
    'score': test_results['y_scores'],
})

predictions_path = os.path.join(exp_dir, 'predictions_test.csv')
df_predictions.to_csv(predictions_path, index=False)
print(f"Predictions saved to {predictions_path}")